# Notebook 2: Creating Embeddings

In this notebook you will learn how to build embeddings from scratch:
sentence → document → topic.

In [1]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

## Outline:
- What is a dense vector?
- What is an "embedding"
- How text is converted into a dense vector (embedding) using a pre-trained model.
- Explanation of "embedding space" and what is happening under the hood...
- Creating a simple embedding function using a pre-trained model from Hugging Face.
- Code to generate a word embedding, a sentence embedding and a document embedding.
    - Understanding the difference between these three types of embeddings.
    - What shape do you get for each type of embedding? Why? What does it mean?
    - Why it makes sense to use sentence embeddings as opppsed to word embeddings for our use case.
- Why do we "preprocess" text before passing it to the embedding model?
     - What does preprocessing do?
     - How do we "clean" text?
     - Why is it important?
     - What do we mean by "structured" document vs and unstructured text
     - What does the "structured" document look like in our case?


## 1. What is an embedding?

A **sparse** representation of "cat" in a vocabulary of 30,000 words is a one-hot vector
that is 0 everywhere and 1 at the index of "cat". It carries no information about *meaning*
— "cat" and "kitten" are as far apart as "cat" and "helicopter".

A **dense** embedding is a fixed-length list of floats — typically 384, 768, or 1024 values —
where every dimension carries a slice of meaning learned from a large corpus.
When we pass text to an embedding model such as 
`all-MiniLM-L6-v2` (the model we use throughout this workshop), it is
transformed into a numeric representation of the term (or terms) that carries associations
and patterns about how that term is used:

```
cat = [-0.0237, 0.0815, -0.0011, …, 0.0532]   # 384 floats
```

Two pieces of text whose meanings are similar end up close together in this 384-dimensional
space, even when they share no tokens. That is what makes semantic search work.

In [2]:
from sentence_transformers import SentenceTransformer

# Same model used by src/preprocess.py and src/search.py (see src/config.py)
model = SentenceTransformer("all-MiniLM-L6-v2")

word     = "cat"
sentence = "The cat is sleeping on the keyboard."
document = (
    "The orange tabby cat napped on the warm keyboard for hours, occasionally "
    "swatting at the cursor before settling back into a contented purr. "
    "The owner gave up trying to type and made a cup of tea instead."
)

for label, text in [("word", word), ("sentence", sentence), ("document", document)]:
    embedding = model.encode(text, convert_to_numpy=True)
    print(f"{label:9s} shape={embedding.shape}  first 5 dims={embedding[:5]}")

word      shape=(384,)  first 5 dims=[ 0.0373303   0.05116175 -0.00030603  0.06020985 -0.11749438]
sentence  shape=(384,)  first 5 dims=[ 0.08553991 -0.02352979  0.00437254  0.04401119 -0.05257662]
document  shape=(384,)  first 5 dims=[ 0.03127092 -0.07033856  0.0053354   0.0710232   0.00286742]


### Why are all three the same shape?

`all-MiniLM-L6-v2` is a *sentence-transformer*. It is a type of neural network. Internally it tokenizes the input,
runs each token through 6 transformer layers to produce contextual token vectors,
and then **mean-pools** those token vectors into a single 384-dimensional vector that
represents the entire input.

Because of pooling, the output shape is always `(384,)` regardless of whether the
input is a single word or a paragraph. That's exactly what we want for search:
every document, no matter how long, becomes one comparable vector.

For our use case (short social-media posts) sentence-level embeddings are a good
match. For long-form documents you might prefer to chunk into sentences/paragraphs,
embed each chunk, and store them separately.

### Embedding space: similar meanings cluster together

Below we encode three short phrases and compute pairwise cosine similarity. Two of
them are about cats; one is about the stock market. The cat phrases share **no
tokens** but should still score high against each other.

In [3]:
import numpy as np
import pandas as pd

phrases = ["cat purring", "kitten meowing", "stock market crash"]
embeddings = model.encode(phrases, convert_to_numpy=True)

# Normalize so the dot product equals cosine similarity
normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
similarity_matrix = normed @ normed.T

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=phrases,
    columns=phrases,
)

print("Cosine similarity between phrase embeddings:")
display(similarity_df.round(4))

Cosine similarity between phrase embeddings:


,cat purring,kitten meowing,stock market crash
cat purring,1.0000,0.6071,0.0578
kitten meowing,0.6071,1.0000,0.0775
stock market crash,0.0578,0.0775,1.0000


### What is happening under the hood


beddEmbedding models translate a term (or "token") to a vector that is representative of is usage.
The model does not rely on (1,0) encoding for each word, it is based on a neural
network. When you pass text through the model, the final layer's pooled output
is a numeric vector.

They are called "embedding" models because they embed meaning into a vector
space — they map discrete objects (words, sentences, documents) into continuous
numeric vectors where geometric relationships encode semantic relationships.

Unlike a "sparse" vector, the
embedding vector is a fixed-size vector representation of the input's meaning in
high-dimensional space. Databases will often refer to it as a "dense vector" for
this reason.

## 2. Generate an embedding

### Structured document

Real social-media posts are messy: emojis, mixed case, punctuation, sometimes only an
image. Before we embed, we wrap each raw `Post` into a `PostDocument` (see
`src/data_models.py`). Think of it as a deconstructed post.

 The `PostDocument` can store  document attributes that are part of
 the unstructured text and turn them into structure documents. Examples: 

- `hashtag` - #NotIncludedInSampleData 
- `@mention` - Also not mentioned in our sample data 
- `emojies` - converted to text representation i.e. `happy_face_smiling_wink`
- `author_attributes` - For example, a bio.
- `publication_metadata` - The publisher, publishing date, copyright notice,
  citation reference, etc.
- `image_caption` — populated by a vision model, in our workshop, if the post has an image.
- `doc_embedding` — the 384-d vector that powers semantic search.

... any any other attribute you may want to include as a search attribute.

In [4]:
from datetime import datetime, UTC

import emoji

from src.data_models import PostDocument

raw_text = "🐈" # Cat!

postdoc = PostDocument(
    post_id="demo-1",
    post_author="workshop",
    created_at=datetime.now(UTC).isoformat(),
    modified_at=datetime.now(UTC).isoformat(),
    post_text=raw_text,
)

print("Raw post_text:", postdoc.post_text)
clean_text = postdoc.preprocess_text()
print("Clean post_text:", clean_text)

Raw post_text: 🐈
Clean post_text: cat


### Sentence splitting

As we saw above, the embedding model treats text in much the same way whether it
is a word, a sentence, or a document. However, when comparing embeddings,
especially in search, it is important to consider the level of granularity to
use for your embeddings.

- Use sentence embeddings when your unit of retrieval is a short, focused claim.
- Use document embeddings when your unit of retrieval is the whole document and its
full context. 

In [5]:
import numpy as np

# Typically you would use a more sophisticated sentence splitter.
# This is just a simple demonstration. See code in repo for examples of using
# SpaCy for sentence splitting.
def sentence_splitter(text: str) -> list[str]:
    """A very naive sentence splitter that just splits on periods."""
    return [s.strip() for s in text.split(".") if s.strip()]


raw_text1 = """
The first Supreme Cat Show took place in 1976.[3] Until then the GCCF itself did not organise cat shows, but licensed shows put on by the breed clubs and area clubs affiliated to it. The Supreme Cat Show was devised as a special show, only open to cats which had won an open class at another championship show under GCCF rules, much in the same way that Crufts is only open to winning dogs. The show grew in size each year until it became big enough to be held at the NEC, which has been its home ever since.
"""

postdoc1 = PostDocument(
    post_id="demo-1",
    post_author="workshop",
    created_at=datetime.now(UTC).isoformat(),
    modified_at=datetime.now(UTC).isoformat(),
    post_text=raw_text1,
)

raw_text2 = """
Markets fell sharply after the earnings report. Investors sold tech stocks
throughout the afternoon. Analysts warned about near-term volatility. Investors
regained confidence with the latest jobs report, and the markets rebounded by
the end of the week. Another example of the market trading aggressively
sideways in anticipation of an inversion of the yield curve, which is often seen as a recession signal.
 """
postdoc2 = PostDocument(
    post_id="demo-2",
    post_author="workshop",
    created_at=datetime.now(UTC).isoformat(),
    modified_at=datetime.now(UTC).isoformat(),
    post_text=raw_text2,
)

# 1) Sentence embeddings -> one sentence per embedding
doc1_sentences = sentence_splitter(postdoc1.post_text)
doc2_sentences = sentence_splitter(postdoc2.post_text)

sent_emb_doc1 = model.encode(
    doc1_sentences,
    batch_size=8, # Let's use batching to speed up encoding multiple sentences
    show_progress_bar=True,
    convert_to_numpy=True,
    )
sent_emb_doc2 = model.encode(
    doc2_sentences,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    )


print("doc1_sentences shape:", sent_emb_doc1.shape)
print("doc2_sentences shape:", sent_emb_doc2.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

doc1_sentences shape: (4, 384)
doc2_sentences shape: (5, 384)


Notice the shape of the array is n-sentences x 384; one vector for each
sentence.  

A search for "yield curve" or "GCCF" would yield that sentence or
chunk of the document which is encoded at a finer level of granularity than a
document as a whole. Larger documents are often broken down into sentences or
paragraphs to improve fine grain matching.

To get a representation of the whole document, we average the sentence
embeddings.  

This can be useful for medium sized documents on one topic. You can get a more balanced document summary, especially for long
texts where one long-pass encoding might overemphasize certain regions or hit
token limits.

**NOTE: For this workshop the posts are short. So, we'll just embed each one as
it's own document.**

### Cleaning and preprocessing text

Cleaning the text cleans words, symbols and other attributes into text that can
be tokenized and read by the model.

In [6]:
### Demo Posts for Quick Experimenting ###
# We create a few tiny PostDocument examples manually so you can test
# preprocess_text() and embedding behavior before touching real workshop data.

from datetime import datetime, UTC

postdocs = [
    PostDocument(
        post_id="demo-1",
        post_author="workshop",
        created_at=datetime.now(UTC).isoformat(),
        modified_at=datetime.now(UTC).isoformat(),
        post_text="#HighSpeedRail https://www.californiarailmap.com/",
        image_caption="A photo of a high-speed train speeding through the California countryside, showcasing the potential of modern transportation infrastructure."
    ),
    PostDocument(
        post_id="demo-2",
        post_author="workshop",
        created_at=datetime.now(UTC).isoformat(),
        modified_at=datetime.now(UTC).isoformat(),
        post_text="Today's fog is more than just beautiful; it’s also serving as a reminder to stay prepared and alert during these unpredictable weather events. @KarlTheFog keeps us informed about what to expect, so everyone can enjoy this view."
    ),
    PostDocument(
        post_id="demo-3",
        post_author="workshop",
        created_at=datetime.now(UTC).isoformat(),
        modified_at=datetime.now(UTC).isoformat(),
        post_text="✨ Shout out to Texas! Come on y'all... Share your joy with a smile and laughter every single day! 🧡",
        image_caption="View of the Austin City Limits music hall in downtown Austin, Texas"
    ),
]

print("\nRaw post_texts:")
for post in postdocs:
    print(post.post_text)

print("\nClean post_texts:")
for post in postdocs:
    print(post.preprocess_text())


Raw post_texts:
#HighSpeedRail https://www.californiarailmap.com/
Today's fog is more than just beautiful; it’s also serving as a reminder to stay prepared and alert during these unpredictable weather events. @KarlTheFog keeps us informed about what to expect, so everyone can enjoy this view.
✨ Shout out to Texas! Come on y'all... Share your joy with a smile and laughter every single day! 🧡

Clean post_texts:
highspeedrail httpswwwcaliforniarailmapcom
today's fog is more than just beautiful its also serving as a reminder to stay prepared and alert during these unpredictable weather events karlthefog keeps us informed about what to expect so everyone can enjoy this view
sparkles shout out to texas come on y'all share your joy with a smile and laughter every single day orange_heart


### Exercise ###
Add or remove preprocessing steps to PostDoc.preprocess_text() to clean any
additional issues

In [7]:
# 2) Average sentence embeddings to get a single document embedding representing the whole document
doc1_embedding = np.mean(sent_emb_doc1, axis=0)
doc2_embedding = np.mean(sent_emb_doc2, axis=0)

print("doc1_embedding shape:", doc1_embedding.shape)
print("doc2_embedding shape:", doc2_embedding.shape)

doc1_embedding shape: (384,)
doc2_embedding shape: (384,)


### Generating embeddings



Embeddings are returned as numpy arrays in the same order as the input texts. To
help with evaluation, speed tasks like topic modeling, etc, we often map them
back to the original texts that were passed to the embedding model. 

In the case of this workshop we map them back to the structured `PostDoc`

In [8]:
postdocs

[PostDocument(post_id='demo-1', post_author='workshop', created_at=datetime.datetime(2026, 5, 9, 2, 59, 23, 977771, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 9, 2, 59, 23, 977771, tzinfo=TzInfo(0)), post_text='#HighSpeedRail https://www.californiarailmap.com/', likes=0, image_url=None, generated_topic=None, image_caption='A photo of a high-speed train speeding through the California countryside, showcasing the potential of modern transportation infrastructure.', doc_embedding=[]),
 PostDocument(post_id='demo-2', post_author='workshop', created_at=datetime.datetime(2026, 5, 9, 2, 59, 23, 977771, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 9, 2, 59, 23, 977771, tzinfo=TzInfo(0)), post_text="Today's fog is more than just beautiful; it’s also serving as a reminder to stay prepared and alert during these unpredictable weather events. @KarlTheFog keeps us informed about what to expect, so everyone can enjoy this view.", likes=0, image_url=None, generated_topic=N

In [9]:
# 1. Build the text string the model will see for each post
# extract_embedding_text combines cleaned post_text and image_caption.

def extract_embedding_text(postdoc: PostDocument) -> str:
    # Extract the cleaned text elements to embed
    text = postdoc.preprocess_text()
    # Check if the post has an image caption (i.e. image converted to text)
    caption = (postdoc.image_caption or "").strip()
    if text and caption:
        return f"{text} {caption}"
    return text or caption

# 2. Batch-encode all texts at once. Batching is much faster than calling
#    the model once per post — the GPU/CPU stays warm between texts.
texts = [extract_embedding_text(postdoc) for postdoc in postdocs]
embeddings = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    )

# 3. Attach each embedding back onto its PostDocument.
# Important: The data must be JSON-serializable for Elasticsearch
# Convert numpy → list so the vector is
for postdoc, embedding in zip(postdocs, embeddings, strict=True):
    postdoc.doc_embedding = embedding.tolist()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
# Reference solution for src.preprocess.PreprocessingPipeline.generate_embeddings
# (lifted from solutions/preprocess.py — adapt to the method body in src/preprocess.py)

def generate_embeddings(self, postdocs):
    """Embed all postdocs using the embedding model."""
    # 1. Build the text string the model will see for each post.
    #    extract_embedding_text combines cleaned post_text and image_caption.
    texts = [extract_embedding_text(postdoc) for postdoc in postdocs]

    # 2. Batch-encode all texts at once. Batching is much faster than calling
    #    the model once per post — the GPU/CPU stays warm between texts.
    embeddings = self.embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    # 3. Attach each embedding back onto its PostDocument. Convert numpy → list
    #    so the vector is JSON-serializable for Elasticsearch and disk storage.
    for postdoc, embedding in zip(postdocs, embeddings, strict=True):
        postdoc.doc_embedding = embedding.tolist()

    return postdocs

# Quick sanity check on a single post — confirms the whole pipeline runs.
embedding = model.encode(extract_embedding_text(postdoc), convert_to_numpy=True)
print(f"Single-post embedding shape: {embedding.shape}")

Single-post embedding shape: (384,)


# Exercise: 

Time: 10 minutes

Using what you have learned, replace ## Exercise: implement `generate_embeddings`

**Time:** ~10 minutes editing + 2–10 minutes runtime (next cell)

You'll edit one function in your project source — use your IDE (VS Code, PyCharm) or any text editor.

1. Open **`src/preprocess.py`** in your editor.
2. Find the function **`generate_embeddings`** (around line 163).
3. Inside the `### EXERCISE ###` block, write code that:
   - extracts text for each post via `extract_embedding_text(postdoc)`
   - calls `self.embedding_model.encode(...)` on those texts
   - stores the result on `postdoc.doc_embedding` **as a list** (call `.tolist()` on the numpy array)
4. Save the file.

**Verify just this function before running the full pipeline:**

```
uv run pytest -k TestGenerateEmbeddings
```


### Run the full preprocessing pipeline

This **must** finish before the topic model can be trained.

**Preferred:** open a terminal at the repo root and run

```bash
uv run python -m src.preprocess
```

> **Heads-up on runtime:** ~2 minutes on a fast machine, up to ~10 minutes on a slow one (CPU-only sentence-transformer + BLIP vision model). The first run also downloads model weights (~500 MB).

**Read the output carefully — here's what to look for:**

| Line | Meaning |
|---|---|
| `Embedding 154 postdocs…` followed by a `Batches: 5/5` progress bar | ✅ Your `generate_embeddings` is producing real vectors |
| `Embedding dimension mismatch — Did you convert the np.array output … to a list?` | ⚠️ Your function isn't storing valid embeddings — revisit the exercise |
| `Saved processed_posts.json (154 postdocs).` | ✅ Disk write succeeded |
| `Elasticsearch not available — skipping Elasticsearch storage.` | ℹ️ Fine — the pipeline continues with disk-only output |
| `Preprocessing complete. Run: uv run python -m src.topic_model` | ✅ Done. **Note this command — you'll run it in the next section.** |

**Fallback (if you ran out of time):**

```bash
uv run python -m solutions.preprocess
```

# Optional: run preprocessing from inside this notebook instead of a terminal.
# Uses YOUR edited src/preprocess.py.
# Expect 2-10 minutes; watch for "Embedding dimension mismatch" warnings.
from src.preprocess import PreprocessingPipeline
PreprocessingPipeline().run()

In [ ]:
# Fallback: run the answer key instead.
# Use this if your generate_embeddings isn't ready and you want to keep moving.
from solutions.preprocess import PreprocessingPipeline as SolutionPipeline
SolutionPipeline().run()

In [ ]:
### Train the topic model

Now train BERTopic on the embeddings you just produced:

```bash
uv run python -m src.topic_model
```

> **Runtime:** another ~2–10 minutes — most of it is the local LLM (Ollama) labeling each topic.

**What to look for:**

| Line | Meaning |
|---|---|
| `Loaded 154 posts from disk.` | ✅ Preprocessing output found |
| `Training on 154 posts.` | ✅ Embeddings present |
| `Training on 0 posts.` then `TypeError: …only contains strings` | ⚠️ Embeddings are missing/empty — go back and fix `generate_embeddings`, then re-run preprocessing |
| BERTopic stages: `Dimensionality → Cluster → Representation` | ✅ Normal training flow |
| `Representation` progress bar `0/8 → 8/8` | The LLM-labeling step — slowest part |

In [ ]:
# Optional: train from inside the notebook.
from src.topic_model import TopicModeler
TopicModeler().run()